**Load features**

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader
import gc 
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

In [11]:
train_features = np.load("features\\train_data.npy")
test_features = np.load("features\\test_data.npy")
val_features = np.load("features\\val_data.npy")

train_labels = np.load("features\\train_labels.npy")
test_labels = np.load("features\\test_labels.npy")
val_labels = np.load("features\\val_labels.npy")

# Transform
encoder = OneHotEncoder(sparse_output=False)
train_labels = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels = encoder.transform(test_labels.reshape(-1, 1))
val_labels = encoder.transform(val_labels.reshape(-1, 1))

standardScaler = StandardScaler()
train_features = standardScaler.fit_transform(train_features)
test_features = standardScaler.transform(test_features)
val_features = standardScaler.transform(val_features)

print(train_features.shape)
print(test_features.shape)
print(val_features.shape)
print(train_labels.shape)
print(test_labels.shape)
print(val_labels.shape)

(4140, 162)
(143, 162)
(259, 162)
(4140, 8)
(143, 8)
(259, 8)


**Neural network architecture and training:**

In [32]:
class BuildResnet18(nn.Module):
    def __init__(self, num_classes):
        super(BuildResnet18, self).__init__()

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Freeze tất cả
        for param in resnet.parameters():
            param.requires_grad = False

        # Unfreeze layer4 và avgpool
        for param in resnet.layer4.parameters():
            param.requires_grad = True

        self.features = nn.Sequential(*list(resnet.children())[:-1])

        self.fc = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, num_classes)
        )

    def forward(self, images):
        x = self.features(images)
        x = x.flatten(1)
        return self.fc(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [26]:
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx]

**Upload processed spectrogram images & create DataLoader**

In [27]:
# Encode labels thành số
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Load CSV ảnh
train_csv = pd.read_csv("CSVs\\train_images.csv")
val_csv   = pd.read_csv("CSVs\\val_images.csv")
test_csv  = pd.read_csv("CSVs\\test_images.csv")

# Fit encoder trên train, transform tất cả
le.fit(train_csv["emotion"])
train_labels = le.transform(train_csv["emotion"])
val_labels   = le.transform(val_csv["emotion"])
test_labels  = le.transform(test_csv["emotion"])

# Tạo Dataset
train_dataset = ImageDataset(train_csv["path"].tolist(), train_labels)
val_dataset   = ImageDataset(val_csv["path"].tolist(),   val_labels)
test_dataset  = ImageDataset(test_csv["path"].tolist(),  test_labels)

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [30]:
num_classes = len(le.classes_)
resnet = BuildResnet18(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(resnet.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()


def evaluate(loader):
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average="weighted")
    prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)

    return avg_loss, acc, f1, prec, rec

## TRAIN / VALIDATE

In [31]:
if "model" in dir():
    del resnet
torch.cuda.empty_cache()
gc.collect()



best_val_loss = float("inf")
patience = 5
no_improve = 0

for epoch in range(100):
    # ── Train ──
    resnet.train()
    train_preds = []
    train_labels_list = []
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels_list.extend(labels.cpu().numpy())

    train_avg_loss = train_loss / len(train_loader)
    train_acc = accuracy_score(train_labels_list, train_preds)
    train_f1 = f1_score(train_labels_list, train_preds, average="weighted")
    train_prec = precision_score(
        train_labels_list, train_preds, average="weighted", zero_division=0
    )
    train_rec = recall_score(
        train_labels_list, train_preds, average="weighted", zero_division=0
    )

    # ── Validation ──
    resnet.eval()
    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(val_loader)

    print(
        f"Epoch {epoch + 1:3d} | "
        f"Train Loss: {train_avg_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} | Train Prec: {train_prec:.4f} | Train Rec: {train_rec:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | Val Prec: {val_prec:.4f} | Val Rec: {val_rec:.4f}"
    )

    # ── Early Stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(resnet.state_dict(), "best_model.pth")
        print(f"  → Saved best model (val_loss: {best_val_loss:.4f})")
    else:
        no_improve += 1
        print(f"  → No improve {no_improve}/{patience}")
        if no_improve >= patience:
            print(f"Early stopping tại epoch {epoch + 1}")
            break

# Load lại model tốt nhất
resnet.load_state_dict(torch.load("best_model.pth"))
print("Loaded best model")

Epoch   1 | Train Loss: 2.1561 | Train Acc: 0.1278 | Train F1: 0.1038 | Train Prec: 0.1297 | Train Rec: 0.1278 | Val Loss: 2.0741 | Val Acc: 0.1622 | Val F1: 0.1405 | Val Prec: 0.1511 | Val Rec: 0.1622
  → Saved best model (val_loss: 2.0741)
Epoch   2 | Train Loss: 2.1029 | Train Acc: 0.1365 | Train F1: 0.1299 | Train Prec: 0.1343 | Train Rec: 0.1365 | Val Loss: 2.0740 | Val Acc: 0.1660 | Val F1: 0.1446 | Val Prec: 0.1506 | Val Rec: 0.1660
  → Saved best model (val_loss: 2.0740)
Epoch   3 | Train Loss: 2.0838 | Train Acc: 0.1476 | Train F1: 0.1427 | Train Prec: 0.1391 | Train Rec: 0.1476 | Val Loss: 2.0655 | Val Acc: 0.1583 | Val F1: 0.1405 | Val Prec: 0.1508 | Val Rec: 0.1583
  → Saved best model (val_loss: 2.0655)
Epoch   4 | Train Loss: 2.0722 | Train Acc: 0.1524 | Train F1: 0.1479 | Train Prec: 0.1465 | Train Rec: 0.1524 | Val Loss: 2.0672 | Val Acc: 0.1737 | Val F1: 0.1522 | Val Prec: 0.1609 | Val Rec: 0.1737
  → No improve 1/5
Epoch   5 | Train Loss: 2.0620 | Train Acc: 0.1621 | 

KeyboardInterrupt: 